In [ ]:
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
from langchain_core.runnables import RunnableWithMessageHistory
from langchain_groq import ChatGroq
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.output_parsers import StrOutputParser
load_dotenv()
store = {}

def get_history(session_id : str):
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

prompt = ChatPromptTemplate([
    ('system','You are a rude assistant which talk in very rude way'),
    MessagesPlaceholder(variable_name='history'),
    ('human','{query}')
])
LLM = ChatGroq(model='openai/gpt-oss-20b',temperature=0.5)
chain = prompt | LLM | StrOutputParser()

chatbot = RunnableWithMessageHistory(
    chain,
    get_history,
    input_messages_key='query',
    history_messages_key='history'

)
res = LLM.invoke('Who is the CEO of OpenAI')
res

C:\Users\shrey\AppData\Local\Temp\ipykernel_7620\902659195.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import ChatMessageHistory
d:\miniconda3\envs\genai\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)
d:\miniconda3\envs\genai\Lib\site-packages\langsmith\_internal\_addressing.py:104: LangSmithBetaWarning: Addressing runs with `langsmith.address` is in beta and is enabled per workspace. A workspace without it rejects the run, so the trace is lost rather than falling back to a project. The behavior may change without notice.
  _warn_once(


In [10]:
# RAG Backend Pipeline
path = './defoe-plague.pdf'
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_classic.retrievers import MultiQueryRetriever
from langchain_core.runnables import RunnablePassthrough
prompt_ret = ChatPromptTemplate([
    ('system','You have to answer the question from this context only, If the context is not sufficient say I do not Know, {context}'),
    ('human', '{query}')
])
docs = PyPDFLoader(path).load()
splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=50)
chunks = splitter.split_documents(docs)
embedding = HuggingFaceEmbeddings(model='sentence-transformers/all-MiniLM-L6-v2')
vector_store = FAISS.from_documents(chunks,embedding)
retriever = MultiQueryRetriever.from_llm(retriever=vector_store.as_retriever(),llm=LLM)
def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

chain_ret = {
    'context': retriever | format_docs,
    'query': RunnablePassthrough()
} | prompt_ret | LLM | StrOutputParser()


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3332.36it/s]


In [17]:
res_ret = chain_ret.invoke('In which country there was Plague')

In [18]:
res_ret

'The passage describes the outbreak of the plague in **England – particularly in London** (and throughout the rest of the kingdom). It also notes that the disease was carried to other countries such as Portugal (the Port of Faro) and Spain, but the primary focus of the narrative is on the plague in England.'

Task 11: Observations & Insights Write short answers: 
1. Difference between PDF Q&A and conversational PDF Q&A?
Conversational PDF Q&A has memory in it. Which can use old msg context.
2. Role of message history in follow-up questions?
It plays a important role if past message context is needed for the current query
3. Trade-offs between long memory and performance?
long memory take more time to process and uses more tokens
4. How trimming history affects answer quality?
It make sure the context length is preserved in any conversation